# GPT from scratch

Watch Karpathy's "Let's build GPT: from scratch, in code, spelled out." alongside this notebook.

The model you'll end up with, top to bottom:

```
characters -> token ids
   -> token embedding + position embedding
   -> N x [ LayerNorm -> multi-head self-attention -> add back (residual)
            LayerNorm -> feed-forward MLP          -> add back (residual) ]
   -> final LayerNorm -> linear layer -> logits over the vocabulary
```

Every piece of that diagram gets built below.

In [ ]:
import math
import os
import time
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)
_ = torch.manual_seed(1337)

## 1. The data and a character tokenizer

In [ ]:
os.makedirs("data", exist_ok=True)
DATA_PATH = "data/input.txt"
if not os.path.exists(DATA_PATH):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, DATA_PATH)

with open(DATA_PATH, encoding="utf-8") as f:
    text = f.read()

print(f"{len(text):,} characters")
print(text[:500])

The simplest possible tokenizer: every distinct character is one token. GPT-2 uses about 50,000 subword tokens instead (module 04). Characters keep the vocabulary tiny and the code simple, at the cost of longer sequences.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print("vocab size:", vocab_size)
print(repr("".join(chars)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print(encode("hii there"))
print(decode(encode("hii there")))

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"train {len(train_data):,} tokens, val {len(val_data):,} tokens")

## 2. Context and targets

The model trains on chunks of `block_size` tokens. One chunk holds `block_size` training examples at once: at every position, the target is the next character, and the context is everything up to and including the current one.

In [ ]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context, target = x[:t + 1], y[t]
    print(f"context {decode(context.tolist())!r:14} -> target {decode([target.item()])!r}")

Training on every context length from 1 to `block_size` also means the model can generate starting from a single character.

Batches stack several random chunks so the GPU processes them in parallel.

In [ ]:
def get_batch(split, batch_size, block_size):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", batch_size=4, block_size=8)
print("inputs", tuple(xb.shape)); print(xb)
print("targets", tuple(yb.shape)); print(yb)

## 3. Baseline: the bigram model

The simplest language model predicts the next character from only the current one, using a lookup table: row `i` holds the logits for what follows character `i`. It ignores all earlier context.

In [ ]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.table(idx)                              # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)          # only the last position predicts the future
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


bigram = BigramLM(vocab_size).to(device)
_, loss = bigram(xb, yb)
print(f"initial loss {loss.item():.3f}   expected for a uniform guess: -ln(1/{vocab_size}) = {math.log(vocab_size):.3f}")

start_ids = torch.zeros((1, 1), dtype=torch.long, device=device)    # token 0 is the newline character
print(decode(bigram.generate(start_ids, 200)[0].tolist()))

In [ ]:
opt = torch.optim.AdamW(bigram.parameters(), lr=1e-2)
for step in range(3000):
    xb, yb = get_batch("train", 32, 8)
    _, loss = bigram(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0 or step == 2999:
        print(f"step {step:4}  loss {loss.item():.3f}")

print(decode(bigram.generate(start_ids, 300)[0].tolist()))

The loss drops from about 4.7 (a bit worse than a uniform guess at first, because a randomly initialized table isn't uniform) to about 2.5, and the output starts to look like words and line breaks, but it's nonsense, because each character only knows about the one before it. To do better, tokens need to gather information from earlier tokens. That's what attention does.

## 4. The averaging trick

Start with the dumbest way for a token to use its past: average the vectors of all tokens up to and including itself. Here are three ways to compute exactly that.

In [ ]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

# Version 1: explicit loops
xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xbow[b, t] = x[b, :t + 1].mean(dim=0)

# Version 2: a lower-triangular weight matrix and one matmul
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x                        # (T, T) @ (B, T, C) -> (B, T, C) by broadcasting

# Version 3: masking with -inf, then softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T).masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x

print(torch.allclose(xbow, xbow2, atol=1e-6), torch.allclose(xbow, xbow3, atol=1e-6))
print(wei)

Version 3 is the shape of attention. Each row says how much token `t` takes from each earlier token. The `-inf` entries become 0 after softmax, so no token can look into the future. Here the weights are uniform. In self-attention, each token computes the weights itself, from the data.

## 5. Self-attention

Every token produces three vectors through learned linear layers:

- a query: "what am I looking for?"
- a key: "what do I contain?"
- a value: "what will I share if you attend to me?"

The affinity between token `t` and an earlier token `s` is the dot product of `t`'s query with `s`'s key. Those affinities replace the zeros in version 3.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k, q, v = key(x), query(x), value(x)                 # each (B, T, 16)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5    # (B, T, 16) @ (B, 16, T) -> (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
out = wei @ v                                        # (B, T, T) @ (B, T, 16) -> (B, T, 16)

print(out.shape)
print(wei[0].detach().numpy().round(2))

Each row still sums to 1 and is still lower-triangular, but now the weights differ from row to row and depend on the content. That's the difference between averaging and attending.

Why multiply by `head_size ** -0.5`? With random inputs, a dot product of two 16-dimensional vectors has a variance of about 16. Large values make softmax extremely peaked, so each token attends to just one other token and gradients vanish. Scaling brings the variance back to about 1.

In [ ]:
q = torch.randn(B, T, head_size)
k = torch.randn(B, T, head_size)
print("variance of q·k without scaling:", (q @ k.transpose(-2, -1)).var().item())
print("variance of q·k with scaling:   ", (q @ k.transpose(-2, -1) * head_size ** -0.5).var().item())

scores = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print("softmax(scores):     ", F.softmax(scores, dim=-1).numpy().round(3))
print("softmax(scores * 8): ", F.softmax(scores * 8, dim=-1).numpy().round(3))

A few facts about attention worth stating plainly:

- Attention has no built-in sense of order. It treats the context as a set. Position information has to be added separately, which is what position embeddings do below.
- The `tril` mask makes this a decoder: tokens can't see the future. A model that reads a whole sentence at once (like BERT, used by the sentiment pipeline in module 04) simply omits the mask.
- This is self-attention because queries, keys, and values all come from the same tokens. In cross-attention, keys and values come from somewhere else, such as an image encoder.

## 6. The full transformer

Hyperparameters first. The defaults train in a few minutes on your Mac. The commented `bigger` settings produce noticeably better text but take several times longer.

In [ ]:
config = dict(
    batch_size=32,
    block_size=128,     # context length in characters
    n_embd=192,         # size of each token's vector
    n_head=6,           # head size = n_embd / n_head = 32
    n_layer=4,          # transformer blocks
    dropout=0.1,
    lr=1e-3,
    max_iters=3000,
    eval_interval=250,
    eval_iters=50,
)
# bigger = dict(batch_size=64, block_size=256, n_embd=384, n_head=6, n_layer=6, dropout=0.2, lr=3e-4, max_iters=5000)
# config.update(bigger)
print(config)

In [ ]:
class Head(nn.Module):
    """One head of masked self-attention."""

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))   # not a parameter
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ self.value(x)


class MultiHeadAttention(nn.Module):
    """Several heads in parallel, each attending to different things, then mixed back together."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Per-token computation: each token thinks about what it gathered."""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Communication (attention) then computation (feed-forward), each wrapped in a residual connection."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout, **_):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding(idx)                                       # (B, T, n_embd)
        pos = self.position_embedding(torch.arange(T, device=idx.device))     # (T, n_embd)
        x = self.blocks(tok + pos)
        logits = self.lm_head(self.ln_f(x))                                   # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]          # position embeddings only go up to block_size
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                kth = torch.topk(logits, top_k).values[:, -1, None]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

Three details in there matter more than they look:

- Residual connections (`x = x + ...`). Each block adds a correction to the running vector instead of replacing it. Gradients flow straight back through the additions, which is what makes deep stacks trainable.
- LayerNorm before each sublayer ("pre-norm") keeps activations at a stable scale. Modern LLMs do the same, often with a variant called RMSNorm.
- The feed-forward layer is 4 times wider than the embedding. Across the model, these MLPs hold about two-thirds of the parameters.

In [ ]:
@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(cfg["eval_iters"])
        for i in range(cfg["eval_iters"]):
            X, Y = get_batch(split, cfg["batch_size"], cfg["block_size"])
            losses[i] = model(X, Y)[1].item()
        out[split] = losses.mean().item()
    model.train()
    return out

torch.manual_seed(1337)
model = GPT(vocab_size, **config).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
print("initial losses:", estimate_loss(model, config), f"(expect about {math.log(vocab_size):.2f})")

## 7. Train

This logs to W&B (project `ml-lab-gpt`) the same way module 05 did. Set `USE_WANDB = False` to skip it.

In [ ]:
import wandb

USE_WANDB = True
run = wandb.init(project="ml-lab-gpt", name="char-gpt", config=config) if USE_WANDB else None

optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])
history = []
start = time.time()

for it in range(config["max_iters"] + 1):
    if it % config["eval_interval"] == 0:
        losses = estimate_loss(model, config)
        history.append((it, losses["train"], losses["val"]))
        elapsed = time.time() - start
        print(f"step {it:5}  train {losses['train']:.4f}  val {losses['val']:.4f}  ({elapsed:.0f}s)")
        if run:
            run.log({"train/loss": losses["train"], "val/loss": losses["val"]}, step=it)
    if it == config["max_iters"]:
        break

    xb, yb = get_batch("train", config["batch_size"], config["block_size"])
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

train_time = time.time() - start
tokens_seen = config["max_iters"] * config["batch_size"] * config["block_size"]
print(f"\ntrained on {tokens_seen:,} tokens in {train_time:.0f}s")
if run:
    run.finish()

In [ ]:
steps, tr, va = zip(*history)
plt.plot(steps, tr, label="train"); plt.plot(steps, va, label="val")
plt.axhline(math.log(vocab_size), color="gray", ls=":", label="uniform guess")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Character-level GPT")
plt.show()

Expect validation loss around 1.6 with the default config (and around 1.5 or lower with the bigger one). Compare to 2.5 for the bigram model. If train loss keeps dropping while val loss flattens or rises, you're seeing the overfitting pattern from module 05.

## 8. Generate

In [ ]:
model.eval()
print(decode(model.generate(start_ids, 600, temperature=0.8)[0].tolist()))

It invents character names, speaker labels, line breaks, and plausible old English, all learned from raw characters. It isn't coherent, because a 2M-parameter model trained for a few minutes can only learn local patterns.

The same temperature and top-k knobs from module 01, on your own model:

In [ ]:
for T in [0.5, 1.0, 1.5]:
    torch.manual_seed(0)
    sample = model.generate(start_ids, 250, temperature=T)[0].tolist()
    print(f"===== temperature {T} =====")
    print(decode(sample))
    print()

torch.manual_seed(0)
print("===== temperature 1.0, top_k 5 =====")
print(decode(model.generate(start_ids, 250, temperature=1.0, top_k=5)[0].tolist()))

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
torch.save({"model_state": model.state_dict(), "config": config, "chars": chars}, "checkpoints/char_gpt.pt")
print("saved checkpoints/char_gpt.pt")

## 9. Scale, for perspective

| Model | Parameters | Training tokens |
|---|---|---|
| Your model | ~2M | ~12M (3,000 steps of 32 x 128) |
| GPT-2 small (2019) | 124M | ~10B |
| GPT-3 (2020) | 175B | 300B |
| Modern open 7B to 8B models | 7B to 8B | 10T to 15T+ |

Same architecture family the whole way up. The differences are scale, data quality, a subword tokenizer instead of characters, and engineering (mixed precision, fused attention kernels, training across thousands of GPUs). The Chinchilla scaling result says a compute-efficient model should see roughly 20 training tokens per parameter. Modern small models go far past that because they're cheaper to run once trained.

The next exercise set ends with byte-pair encoding, the tokenizer algorithm that GPT-2, Llama, and Qwen all use.

---
# Exercises

**Exercise 1: Remove the scaling.** Build two attention-score matrices from random queries and keys with `head_size = 64`: one scaled by `head_size ** -0.5` and one not. Apply the causal mask and softmax to each, and compare the average of each row's largest weight. What does the unscaled version do to how attention is spread, and what would that do to learning?

**Exercise 2: Remove the position embeddings.** Make a subclass `GPTNoPos` whose `forward` skips the position embedding. Train it and a normal `GPT` for 1,000 steps each with the same config and seed, and compare validation loss. Explain the result using what you know about attention.

**Exercise 3: Byte-pair encoding.** Implement BPE on the first 100,000 characters of the text:
1. Start from the UTF-8 bytes of the text (ids 0 to 255).
2. `get_stats(ids)`: count how often each adjacent pair occurs.
3. `merge(ids, pair, new_id)`: replace every occurrence of `pair` with `new_id`.
4. Repeat 200 times, merging the most frequent pair each time, and record the merges.

Print the compression ratio (bytes / tokens), the first 20 merges as text, and write `encode`/`decode` functions that round-trip a sentence.

**Exercise 4: Count parameters by component.** Using `named_parameters()`, sum the parameters in the embeddings, attention layers, feed-forward layers, LayerNorms, and the output head. Which component dominates? Then estimate training compute with the rule of thumb FLOPs ≈ 6 × parameters × tokens, and use your measured training time to work out how many FLOPs per second your Mac actually achieved.

**Exercise 5: Your own corpus.** Replace `data/input.txt` with a different text of at least about 1 MB (a public-domain book from Project Gutenberg, a pile of code from one of your repos, or anything else). Rerun the notebook from section 1 and see what your model learns to imitate.

---
# Solutions

In [ ]:
# Exercise 1
torch.manual_seed(0)
hs, T = 64, 128
q, k = torch.randn(32, T, hs), torch.randn(32, T, hs)
mask = torch.tril(torch.ones(T, T)) == 0
for name, scale in [("scaled", hs ** -0.5), ("unscaled", 1.0)]:
    w = F.softmax((q @ k.transpose(-2, -1) * scale).masked_fill(mask, float("-inf")), dim=-1)
    print(f"{name:9} average max weight per row: {w.max(dim=-1).values[:, 16:].mean():.3f}")
# Unscaled scores have variance ~64, so softmax collapses toward one-hot: each token attends to a
# single other token, and the softmax gradient is near zero almost everywhere, so learning stalls.

In [ ]:
# Exercise 2
class GPTNoPos(GPT):
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.blocks(self.token_embedding(idx))          # no position information at all
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

def quick_train(model_cls, steps=1000):
    torch.manual_seed(1337)
    m = model_cls(vocab_size, **config).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=config["lr"])
    for _ in range(steps):
        xb, yb = get_batch("train", config["batch_size"], config["block_size"])
        _, loss = m(xb, yb)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return estimate_loss(m, config)["val"]

print("with positions:   ", quick_train(GPT))
print("without positions:", quick_train(GPTNoPos))
# Without position embeddings, attention sees the context as an unordered bag of characters.
# The causal mask still leaks a little order (each position sees a different-sized set),
# but the model can't tell "th" from "ht", so its loss is clearly worse.

In [ ]:
# Exercise 3
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

sample_text = text[:100_000]
ids = list(sample_text.encode("utf-8"))
n_bytes = len(ids)
merges = {}                                   # (a, b) -> new id
vocab = {i: bytes([i]) for i in range(256)}   # id -> the bytes it stands for

for i in range(200):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    new_id = 256 + i
    ids = merge(ids, pair, new_id)
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]

print(f"compression: {n_bytes:,} bytes -> {len(ids):,} tokens ({n_bytes / len(ids):.2f}x)")
print("first 20 merges:", [vocab[256 + i].decode("utf-8", errors="replace") for i in range(20)])

def bpe_encode(s):
    ids = list(s.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))   # apply merges in learned order
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

def bpe_decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

s = "First Citizen: Before we proceed any further, hear me speak."
enc = bpe_encode(s)
print(len(s), "chars ->", len(enc), "tokens; round trip ok:", bpe_decode(enc) == s)

In [ ]:
# Exercise 4
groups = {"embeddings": 0, "attention": 0, "feed-forward": 0, "layernorm": 0, "output head": 0}
for name, p in model.named_parameters():
    if "embedding" in name:
        groups["embeddings"] += p.numel()
    elif ".attn." in name:
        groups["attention"] += p.numel()
    elif ".ffwd." in name:
        groups["feed-forward"] += p.numel()
    elif "ln" in name:
        groups["layernorm"] += p.numel()
    else:
        groups["output head"] += p.numel()

total = sum(groups.values())
for k_, v_ in groups.items():
    print(f"{k_:13} {v_:>10,}  {v_ / total:5.1%}")

flops = 6 * total * tokens_seen
print(f"\ntraining compute ≈ 6 x {total:,} x {tokens_seen:,} = {flops:.2e} FLOPs")
print(f"achieved ≈ {flops / train_time:.2e} FLOP/s (rough: train_time also includes the evaluation passes)")

Exercise 5 has no code solution: swap the file, delete the cached `data/input.txt` if you're downloading a new one, and rerun from section 1. Code makes a fun corpus, because the model learns indentation and bracket matching surprisingly quickly.